# 14B. 중단된 CNN 대상별 탐색

MusicGen·Udio 각각에 36개 후보를 탐색하려던 계획은 중단됐다. MusicGen의 첫 trial도 최고 epoch와 임계값을 확정하기 전에 종료돼 완료 후보가 아니다. 이후 분석은 [고정 설정 전이 프로토콜](../docs/UNSEEN_FIXED_PROTOCOL.md)을 따른다.

## 1. 중단 상태와 보존 파일 확인

다음 셀은 `aborted_search/metadata.json`과 반올림된 console epoch log만 읽는다. 기존 모델 학습을 이어가지 않고 Test score도 열지 않는다. Checkpoint hash·최고 저장 epoch를 확인해 우발적 변조를 감지한다.

In [1]:
# 중단 시점의 trial 로그와 checkpoint만 보존하고 추가 탐색·Test 평가는 실행하지 않는다.
from pathlib import Path
import json
from src.cnn_unseen_revised import sha256_file

ROOT = Path.cwd()
RUN_ID = "cnn_unseen_strict_20260919T171232Z"
folder = ROOT / "results/cnn_unseen_revised" / RUN_ID / "aborted_search"
metadata = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
checkpoint = Path(metadata["checkpoint_path"])
# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert sha256_file(checkpoint) == metadata["checkpoint_sha256"]
assert (
    metadata["status"] == "aborted_search"
    and not metadata["completed_trial_csv_present"]
)
assert not metadata["test_scored"]
print("status:", metadata["status"], "| trial:", metadata["trial_status"])
print(
    "logged epochs:",
    metadata["logged_epochs"],
    "| interrupted during epoch:",
    metadata["interrupted_during_epoch"],
)
print("saved interim checkpoint epoch:", metadata["checkpoint_saved_best_epoch"])
print("checkpoint SHA-256:", metadata["checkpoint_sha256"])
print(
    "interim Val Track EER/AUC:",
    metadata["checkpoint_val_track_eer"],
    metadata["checkpoint_val_track_roc_auc"],
)
print("console log:", folder / "console_epoch_log.txt")

status: aborted_search | trial: interrupted_before_completion
logged epochs: 11 | interrupted during epoch: 12
saved interim checkpoint epoch: 10
checkpoint SHA-256: 871675884e7b46aa9ca05a0e539b958f2f845b415598760f5fedaf26d390c7f6
interim Val Track EER/AUC: 0.11136363636363633 0.9710227272727273
console log: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/results/cnn_unseen_revised/cnn_unseen_strict_20260919T171232Z/aborted_search/console_epoch_log.txt


MusicGen Baseline의 console 기록은 epoch **1–11**까지 남았다. 사용자 요청으로 epoch **12 중** 프로세스를 중단했다. 보존 checkpoint는 실행 중 저장된 **epoch 10** 상태이며 당시 Validation Track EER **0.1113636364**, AUC **0.9710227273**이다. 완료 trial CSV와 Test score는 **없다**.

이 수치는 중단 전의 임시 epoch 지표이며, 후보 학습 완료 후 최고 checkpoint 복원·Validation threshold 결정까지 끝낸 결과가 아니다. `aborted_search`로만 보관한다.

console epoch log는 화면에 표시된 소수 넷째 자리의 반올림 값만 복구했다. 정확한 epoch별 전체 history는 SIGINT 전에 저장되지 않았다. 이 임시 checkpoint와 36개 후보 계획을 새 primary 성능으로 인용하거나 선택에 사용하지 않는다.